In [ ]:
# =========================
# fMRI Viewer - full rewrite
# =========================

import math
import os
from functools import lru_cache

import nibabel as nib
import numpy as np
import plotly.express as px
from dash import ALL, Dash, Input, Output, State, dcc, html
from dash.exceptions import PreventUpdate


# =========================
# 1) CONFIG
# =========================
BASE_DIR = "/Users/cassidyschuman/Downloads/Classes/Senior Spring/Clinical Preceptorship/example_output"
# BASE_DIR = "/Users/forrestlin/Downloads/example_output"

AXIS_LABELS = {
    0: "Sagittal",
    1: "Coronal",
    2: "Axial",
}

NETWORK_COLORS = [
    np.array([0.92, 0.34, 0.34]),  # red
    np.array([0.24, 0.62, 0.91]),  # blue
    np.array([0.38, 0.77, 0.34]),  # green
    np.array([0.95, 0.76, 0.27]),  # yellow
    np.array([0.66, 0.45, 0.91]),  # purple
    np.array([0.22, 0.78, 0.73]),  # teal
    np.array([0.95, 0.48, 0.66]),  # pink
    np.array([0.67, 0.55, 0.28]),  # brown
    np.array([0.55, 0.64, 0.75]),  # slate
    np.array([0.86, 0.42, 0.20]),  # orange
    np.array([0.34, 0.53, 0.92]),  # indigo
    np.array([0.62, 0.76, 0.25]),  # lime
]

CARD_STYLE = {
    "backgroundColor": "#f4f4f4",
    "borderRadius": "16px",
    "padding": "20px",
    "boxShadow": "0 1px 6px rgba(0,0,0,0.08)",
}

SECTION_TITLE_STYLE = {
    "fontSize": "18px",
    "fontWeight": "700",
    "marginBottom": "14px",
}

LABEL_STYLE = {
    "fontSize": "14px",
    "fontWeight": "600",
    "marginBottom": "8px",
    "display": "block",
}


# =========================
# 2) DATA DISCOVERY
# =========================
def list_patients():
    if not os.path.isdir(BASE_DIR):
        return []
    return sorted(
        d for d in os.listdir(BASE_DIR)
        if os.path.isdir(os.path.join(BASE_DIR, d))
    )


PATIENTS = list_patients()


def patient_display_dir(patient_id):
    return os.path.join(BASE_DIR, patient_id, "xprx", "display")


def patient_root_dir(patient_id):
    return os.path.join(BASE_DIR, patient_id)


def get_display_niftis(patient_id):
    """
    Returns:
        {
            "reho": "full_filename.nii.gz",
            "seedcorr": "full_filename.nii.gz",
            ...
        }
    """
    display_path = patient_display_dir(patient_id)
    if not os.path.isdir(display_path):
        return {}

    nii_files = [
        f for f in os.listdir(display_path)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ]

    file_dict = {}
    prefix = f"sub-{patient_id}_ses-1_task-rest_acq-multiecho_"

    for f in sorted(nii_files):
        name = f.replace(".nii.gz", "").replace(".nii", "")
        if name.startswith(prefix):
            suffix = name.replace(prefix, "")
        else:
            suffix = name
        file_dict[suffix] = f

    return file_dict


def find_t1_file(patient_id):
    """
    Tries to find a T1/T1w anatomical file anywhere inside the patient folder.
    Adjust the matching rules here if your naming differs.
    """
    root = patient_root_dir(patient_id)
    if not os.path.isdir(root):
        return None

    preferred_terms = [
        "desc-preproc_T1w",
        "T1w",
        "T1",
        "t1w",
        "t1",
    ]

    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
                continue
            full_path = os.path.join(dirpath, fname)
            score = 0
            for i, term in enumerate(preferred_terms):
                if term in fname:
                    score = max(score, len(preferred_terms) - i)
            if score > 0:
                candidates.append((score, full_path))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][1]


# =========================
# 3) VOLUME HELPERS
# =========================
@lru_cache(maxsize=256)
def load_nifti(path):
    img = nib.load(path)
    vol = img.get_fdata().astype(np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]
    vol = np.nan_to_num(vol)
    return vol


@lru_cache(maxsize=256)
def load_patient_overlay(patient_id, filename):
    path = os.path.join(patient_display_dir(patient_id), filename)
    return load_nifti(path)


@lru_cache(maxsize=64)
def load_patient_t1(patient_id):
    t1_path = find_t1_file(patient_id)
    if t1_path is None:
        return None
    return load_nifti(t1_path)


def normalize_to_unit(arr):
    arr = np.asarray(arr, dtype=np.float32)
    vmin = float(np.min(arr))
    vmax = float(np.max(arr))
    if vmax <= vmin:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - vmin) / (vmax - vmin)


def clip_percentile(sl, p_low=1, p_high=99):
    lo, hi = np.percentile(sl, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(sl, dtype=np.float32)
    return np.clip(sl, lo, hi).astype(np.float32)


def get_slice(vol3d, axis, idx):
    if axis == 0:
        sl = vol3d[idx, :, :]
    elif axis == 1:
        sl = vol3d[:, idx, :]
    else:
        sl = vol3d[:, :, idx]
    return np.rot90(sl)


def make_mosaic(vol3d, axis=2, start=0, step=1, n_slices=36, pad=2):
    max_idx = vol3d.shape[axis] - 1
    start = int(np.clip(start, 0, max_idx))
    step = max(1, int(step))
    n_slices = max(1, int(n_slices))

    indices = list(range(start, max_idx + 1, step))[:n_slices]
    if not indices:
        indices = [start]

    slices = [clip_percentile(get_slice(vol3d, axis, i), 1, 99) for i in indices]

    cols = int(math.ceil(math.sqrt(len(slices))))
    rows = int(math.ceil(len(slices) / cols))

    h, w = slices[0].shape
    mosaic_h = rows * h + (rows - 1) * pad
    mosaic_w = cols * w + (cols - 1) * pad
    mosaic = np.zeros((mosaic_h, mosaic_w), dtype=np.float32)

    for k, sl in enumerate(slices):
        r = k // cols
        c = k % cols
        y0 = r * (h + pad)
        x0 = c * (w + pad)
        mosaic[y0:y0 + h, x0:x0 + w] = sl

    return mosaic, indices


def blend_single_slice(base_vol, overlay_vols, axis, idx, threshold, opacities):
    base_slice = get_slice(base_vol, axis, idx)
    base_slice = clip_percentile(base_slice, 1, 99)
    base_slice = normalize_to_unit(base_slice)

    rgb = np.stack([base_slice, base_slice, base_slice], axis=-1)

    for i, vol in enumerate(overlay_vols):
        overlay_slice = get_slice(vol, axis, idx)
        overlay_slice = normalize_to_unit(overlay_slice)

        mask = overlay_slice >= threshold
        if not np.any(mask):
            continue

        alpha = float(np.clip(opacities[i], 0, 1))
        color = NETWORK_COLORS[i % len(NETWORK_COLORS)]

        rgb[mask] = (1.0 - alpha) * rgb[mask] + alpha * color

    return np.clip(rgb, 0, 1)


def build_opacity_controls(selected_files, options):
    if not selected_files:
        return html.Div(
            "Select one or more overlays to tune opacity.",
            style={"color": "#666", "fontSize": "14px"},
        )

    label_lookup = {opt["value"]: opt["label"] for opt in options or []}

    rows = []
    for i, fname in enumerate(selected_files):
        label = label_lookup.get(fname, fname)
        color = NETWORK_COLORS[i % len(NETWORK_COLORS)]
        rgb = tuple((255 * color).astype(int))

        rows.append(
            html.Div(
                style={"marginBottom": "18px"},
                children=[
                    html.Div(
                        style={
                            "display": "flex",
                            "alignItems": "center",
                            "gap": "10px",
                            "marginBottom": "8px",
                        },
                        children=[
                            html.Div(
                                style={
                                    "width": "14px",
                                    "height": "14px",
                                    "borderRadius": "3px",
                                    "backgroundColor": f"rgb{rgb}",
                                    "flexShrink": 0,
                                }
                            ),
                            html.Div(
                                label,
                                style={
                                    "fontWeight": "600",
                                    "fontSize": "14px",
                                    "overflow": "hidden",
                                    "textOverflow": "ellipsis",
                                    "whiteSpace": "nowrap",
                                },
                            ),
                        ],
                    ),
                    dcc.Slider(
                        id={"type": "overlay-opacity", "index": fname},
                        min=0,
                        max=1,
                        step=0.05,
                        value=0.70,
                        marks={0: "0", 0.5: "0.5", 1: "1"},
                        updatemode="drag",
                        tooltip={"placement": "bottom", "always_visible": False},
                    ),
                ],
            )
        )
    return rows


def build_legend(selected_files, options):
    if not selected_files:
        return html.Div("No overlay selected.", style={"color": "#666"})

    label_lookup = {opt["value"]: opt["label"] for opt in options or []}
    items = []

    items.append(
        html.Div(
            style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
            children=[
                html.Div(
                    style={
                        "width": "14px",
                        "height": "14px",
                        "borderRadius": "3px",
                        "backgroundColor": "#cfcfcf",
                    }
                ),
                html.Div("T1 anatomical base", style={"fontSize": "14px", "fontWeight": "600"}),
            ],
        )
    )

    for i, fname in enumerate(selected_files):
        label = label_lookup.get(fname, fname)
        rgb = tuple((255 * NETWORK_COLORS[i % len(NETWORK_COLORS)]).astype(int))
        items.append(
            html.Div(
                style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
                children=[
                    html.Div(
                        style={
                            "width": "14px",
                            "height": "14px",
                            "borderRadius": "3px",
                            "backgroundColor": f"rgb{rgb}",
                        }
                    ),
                    html.Div(label, style={"fontSize": "14px", "fontWeight": "600"}),
                ],
            )
        )
    return items


def empty_figure(message="No image available"):
    fig = px.imshow(np.zeros((16, 16)))
    fig.update_layout(
        template="plotly_white",
        xaxis={"visible": False},
        yaxis={"visible": False},
        coloraxis_showscale=False,
        annotations=[
            dict(
                text=message,
                x=0.5,
                y=0.5,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=18, color="#666"),
            )
        ],
        margin=dict(l=10, r=10, t=10, b=10),
    )
    return fig


# =========================
# 4) APP
# =========================
app = Dash(__name__)
app.title = "fMRI Visualization"

default_patient = PATIENTS[0] if PATIENTS else None

app.layout = html.Div(
    style={
        "backgroundColor": "#ececec",
        "minHeight": "100vh",
        "padding": "16px",
        "fontFamily": "Inter, system-ui, -apple-system, Segoe UI, Roboto, sans-serif",
    },
    children=[
        # Header
        html.Div(
            style={
                **CARD_STYLE,
                "marginBottom": "16px",
                "padding": "22px 28px",
                "display": "flex",
                "justifyContent": "center",
                "alignItems": "center",
            },
            children=[
                html.H1(
                    "fMRI Visualization",
                    style={
                        "margin": 0,
                        "fontSize": "42px",
                        "fontWeight": "800",
                        "letterSpacing": "0.5px",
                    },
                )
            ],
        ),

        # Main grid
        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "300px minmax(700px, 1fr) 320px",
                "gap": "16px",
                "alignItems": "start",
            },
            children=[
                # LEFT COLUMN
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Patient Selection", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Patient", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="patient_dropdown",
                                    options=[{"label": p, "value": p} for p in PATIENTS],
                                    value=default_patient,
                                    clearable=False,
                                ),
                                html.Div(style={"height": "16px"}),
                                html.Label("Processed Output", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="nifti_dropdown",
                                    multi=True,
                                    placeholder="Select overlays",
                                ),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Viewing Controls", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("View Mode", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="mode",
                                    options=[
                                        {"label": "Single slice", "value": "single"},
                                        {"label": "Mosaic", "value": "mosaic"},
                                    ],
                                    value="single",
                                    labelStyle={"display": "block", "marginBottom": "8px"},
                                    inputStyle={"marginRight": "8px"},
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Label("Plane", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="axis",
                                    options=[
                                        {"label": "Sagittal", "value": 0},
                                        {"label": "Coronal", "value": 1},
                                        {"label": "Axial", "value": 2},
                                    ],
                                    value=2,
                                    inline=True,
                                    labelStyle={"marginRight": "16px"},
                                    inputStyle={"marginRight": "6px"},
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Label("Threshold", style=LABEL_STYLE),
                                dcc.Slider(
                                    id="threshold",
                                    min=0,
                                    max=1,
                                    step=0.01,
                                    value=0.50,
                                    marks={0: "0", 0.5: "0.5", 1: "1"},
                                    updatemode="drag",
                                ),
                                html.Div(id="threshold_text", style={"marginTop": "10px", "fontSize": "14px", "color": "#444"}),
                            ],
                        ),
                    ],
                ),

                # CENTER COLUMN
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div(
                                    style={
                                        "display": "flex",
                                        "justifyContent": "space-between",
                                        "alignItems": "center",
                                        "marginBottom": "10px",
                                    },
                                    children=[
                                        html.Div("Image Viewer", style={**SECTION_TITLE_STYLE, "marginBottom": 0}),
                                        html.Div(
                                            id="note",
                                            style={"fontSize": "13px", "color": "#666", "textAlign": "right"},
                                        ),
                                    ],
                                ),
                                dcc.Graph(
                                    id="view",
                                    style={"height": "760px"},
                                    config={"displayModeBar": False},
                                    figure=empty_figure("Select a patient to begin"),
                                ),
                                html.Div(style={"height": "6px"}),
                                html.Label("Slice", style=LABEL_STYLE),
                                dcc.Slider(
                                    id="idx",
                                    min=0,
                                    max=100,
                                    step=1,
                                    value=50,
                                    updatemode="drag",
                                ),
                                html.Div(style={"height": "20px"}),
                                html.Label("Mosaic slices", style=LABEL_STYLE),
                                dcc.Slider(
                                    id="mosaic_n",
                                    min=4,
                                    max=100,
                                    step=4,
                                    value=36,
                                    marks={4: "4", 16: "16", 36: "36", 64: "64", 100: "100"},
                                    updatemode="drag",
                                ),
                            ],
                        ),
                    ],
                ),

                # RIGHT COLUMN
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Network Overlay", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Div(id="network_legend"),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Overlay Opacity", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Div(id="overlay_opacity_controls"),
                            ],
                        ),
                    ],
                ),
            ],
        ),
    ],
)


# =========================
# 5) CALLBACKS
# =========================
@app.callback(
    Output("threshold_text", "children"),
    Input("threshold", "value"),
)
def update_threshold_text(threshold):
    threshold = 0 if threshold is None else float(threshold)
    return f"{threshold:.2f} ({int(round(threshold * 100))}% of max)"


@app.callback(
    Output("nifti_dropdown", "options"),
    Output("nifti_dropdown", "value"),
    Input("patient_dropdown", "value"),
)
def update_nifti_dropdown(patient_id):
    if not patient_id:
        return [], []

    file_dict = get_display_niftis(patient_id)
    if not file_dict:
        return [], []

    options = [{"label": k, "value": v} for k, v in sorted(file_dict.items())]
    default_values = [options[0]["value"]] if options else []
    return options, default_values


@app.callback(
    Output("overlay_opacity_controls", "children"),
    Output("network_legend", "children"),
    Input("nifti_dropdown", "value"),
    State("nifti_dropdown", "options"),
)
def update_overlay_ui(selected_files, options):
    selected_files = selected_files or []
    return build_opacity_controls(selected_files, options), build_legend(selected_files, options)


@app.callback(
    Output("idx", "max"),
    Output("idx", "value"),
    Output("note", "children"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("axis", "value"),
    Input("mode", "value"),
    Input("mosaic_n", "value"),
    prevent_initial_call=False,
)
def sync_axis(patient_id, selected_files, axis, mode, mosaic_n):
    if not patient_id or axis is None:
        raise PreventUpdate

    base_vol = load_patient_t1(patient_id)

    if base_vol is None:
        if not selected_files:
            raise PreventUpdate
        base_vol = load_patient_overlay(patient_id, selected_files[0])

    axis = int(axis)
    max_idx = base_vol.shape[axis] - 1
    mid = max_idx // 2

    note = f"T1 base: {base_vol.shape} | {AXIS_LABELS[axis]} | mode: {mode}"
    if mode == "mosaic":
        note += f" | mosaic_n: {mosaic_n}"

    return max_idx, mid, note


@app.callback(
    Output("view", "figure"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("threshold", "value"),
    Input("mode", "value"),
    Input("axis", "value"),
    Input("idx", "value"),
    Input("mosaic_n", "value"),
    Input({"type": "overlay-opacity", "index": ALL}, "value"),
    State({"type": "overlay-opacity", "index": ALL}, "id"),
)
def update_view(patient_id, selected_files, threshold, mode, axis, idx, mosaic_n, opacity_values, opacity_ids):
    if not patient_id or mode is None or axis is None or idx is None or mosaic_n is None:
        raise PreventUpdate

    threshold = 0.0 if threshold is None else float(threshold)
    axis = int(axis)
    idx = int(idx)
    mosaic_n = int(mosaic_n)

    base_vol = load_patient_t1(patient_id)
    if base_vol is None:
        if not selected_files:
            return empty_figure("No T1 file found and no overlay selected")
        base_vol = load_patient_overlay(patient_id, selected_files[0])

    max_idx = base_vol.shape[axis] - 1
    snapped = max(0, min(idx, max_idx))

    selected_files = selected_files or []
    overlay_vols = [load_patient_overlay(patient_id, f) for f in selected_files]

    opacity_map = {}
    for oid, val in zip(opacity_ids or [], opacity_values or []):
        opacity_map[oid["index"]] = float(val)

    overlay_opacities = [opacity_map.get(fname, 0.70) for fname in selected_files]

    if mode == "single":
        rgb = blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=axis,
            idx=snapped,
            threshold=threshold,
            opacities=overlay_opacities,
        )

        fig = px.imshow(rgb)
        fig.update_layout(
            template="plotly_white",
            margin=dict(l=10, r=10, t=50, b=10),
            title=f"{AXIS_LABELS[axis]} slice {snapped}",
            coloraxis_showscale=False,
        )
        fig.update_xaxes(showticklabels=False)
        fig.update_yaxes(showticklabels=False)
        return fig

    mosaic, indices = make_mosaic(
        base_vol,
        axis=axis,
        start=snapped,
        step=1,
        n_slices=mosaic_n,
        pad=2,
    )

    mosaic_norm = normalize_to_unit(mosaic)
    fig = px.imshow(mosaic_norm, color_continuous_scale="gray")
    fig.update_layout(
        template="plotly_white",
        margin=dict(l=10, r=10, t=50, b=10),
        title=f"Mosaic | {AXIS_LABELS[axis]} | start={snapped} | n={len(indices)}",
        coloraxis_showscale=False,
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig


# =========================
# 6) RUN
# =========================
if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=8051, use_reloader=False)